In [ ]:
fenetre_jours = 5   # premier lancement sur tout l'historique : 1046

In [ ]:
import uuid
from datetime import datetime, timedelta, timezone, date
from zoneinfo import ZoneInfo
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

spark.conf.set("spark.sql.session.timeZone", "UTC")
TZ = ZoneInfo("Europe/Paris")
run_id = str(uuid.uuid4())
run_ts = datetime.now(timezone.utc)

FILIERES = ["nucleaire", "eolien", "solaire", "hydraulique", "gaz", "charbon", "fioul", "bioenergies"]
FLUX = ["ech_physiques", "pompage", "stockage_batterie", "destockage_batterie"]
MESURES = ["consommation", "prevision_j", "prevision_j1", "taux_co2"] + FILIERES + FLUX

In [ ]:
bronze = spark.table("bronze_eco2mix")
mesures_presentes = [c for c in MESURES if c in bronze.columns]

df = (
    bronze
    .withColumn("ts_utc", F.to_timestamp("date_heure"))
    .where(F.col("ts_utc") >= F.current_timestamp() - F.expr(f"INTERVAL {int(fenetre_jours)} DAYS"))
    .withColumn("ts_local", F.from_utc_timestamp("ts_utc", "Europe/Paris"))
    .withColumn("date_locale", F.to_date("ts_local"))
    .withColumn("priorite_nature",
                F.when(F.col("nature").contains("finitive"), 3)
                 .when(F.col("nature").contains("onsolid"), 2)
                 .otherwise(1))
)
for c in mesures_presentes:
    df = df.withColumn(c, F.col(c).cast("double"))

publie = df.where(F.col("consommation").isNotNull())
print("Mesures trouvées :", mesures_presentes)

In [ ]:
w = Window.partitionBy("ts_utc").orderBy(F.desc("priorite_nature"), F.desc("_ingested_at"))
silver_batch = (
    publie.withColumn("_rn", F.row_number().over(w)).where("_rn = 1")
    .select("ts_utc", "ts_local", "date_locale", "nature", "priorite_nature",
            *mesures_presentes, "_ingested_at", "_source_file")
    .withColumn("pas_min",
                F.when(F.max((F.minute("ts_local") % 30 != 0).cast("int"))
                        .over(Window.partitionBy("date_locale")) == 1, 15)
                 .otherwise(30))
    .withColumn("_silver_updated_at", F.current_timestamp())
)

In [ ]:
if not spark.catalog.tableExists("silver_eco2mix"):
    silver_batch.write.saveAsTable("silver_eco2mix")
else:
    (DeltaTable.forName(spark, "silver_eco2mix").alias("t")
        .merge(silver_batch.alias("s"), "t.ts_utc = s.ts_utc")
        .whenMatchedUpdateAll(condition="s.priorite_nature >= t.priorite_nature")
        .whenNotMatchedInsertAll()
        .execute())
print("Silver :", spark.table("silver_eco2mix").count(), "lignes")

In [ ]:
resultats = []
def controle(nom, perimetre, valeur, seuil, ok, detail=""):
    resultats.append((run_id, run_ts.replace(tzinfo=None), nom, str(perimetre),
                      float(valeur), float(seuil), "OK" if ok else "KO", detail))

# a. Doublons dans un même fichier source
dup = (bronze.where(F.col("_ingested_at") >= F.lit(run_ts - timedelta(days=1)))
       .groupBy("_source_file", "date_heure").count().where("count > 1").count())
controle("doublons_fichier_source", "dernier lot", dup, 0, dup == 0)

# b. Complétude par jour (gère le changement d'heure et le pas de 15 ou 30 min)
def attendus(d, pas_min):
    debut = datetime(d.year, d.month, d.day, tzinfo=TZ).astimezone(timezone.utc)
    fin_j = d + timedelta(days=1)
    fin = datetime(fin_j.year, fin_j.month, fin_j.day, tzinfo=TZ).astimezone(timezone.utc)
    return int((fin - debut).total_seconds() // (pas_min * 60))

silver = spark.table("silver_eco2mix")
aujourdhui = datetime.now(TZ).date()
comptes = {r["date_locale"]: (r["n"], r["pas"]) for r in
           silver.where(F.col("date_locale") >= F.lit(aujourdhui - timedelta(days=int(fenetre_jours))))
                 .where(F.col("date_locale") < F.lit(aujourdhui))
                 .groupBy("date_locale")
                 .agg(F.count("*").alias("n"), F.min("pas_min").alias("pas")).collect()}
for i in range(1, int(fenetre_jours)):
    d = aujourdhui - timedelta(days=i)
    n, pas = comptes.get(d, (0, 15))
    att = attendus(d, pas)
    controle("completude_jour", d, n / att, 1.0, n >= att, f"{n}/{att} relevés au pas de {pas} min")

# c. Valeurs manquantes par filière
fen = silver.where(F.col("date_locale") >= F.lit(aujourdhui - timedelta(days=int(fenetre_jours))))
total = fen.count()
if total:
    nulls = fen.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                        for c in FILIERES + ["taux_co2"] if c in fen.columns]).first().asDict()
    for c, n in nulls.items():
        controle("taux_manquant", c, n / total, 0.01, n / total <= 0.01, f"{n} sur {total}")

# d. Bilan : consommation ≈ production + échanges + stockage
prod = sum(F.coalesce(F.col(c), F.lit(0.0)) for c in FILIERES if c in fen.columns)
flux = sum(F.coalesce(F.col(c), F.lit(0.0)) for c in FLUX if c in fen.columns)
ecart = (fen.where(F.col("nucleaire").isNotNull())
            .select((F.abs(F.col("consommation") - (prod + flux)) / F.col("consommation")).alias("e"))
            .agg(F.percentile_approx("e", 0.95).alias("p95")).first()["p95"])
if ecart is not None:
    controle("bilan_p95_ecart_relatif", "fenetre", ecart, 0.02, ecart <= 0.02)

# e. Plausibilité
hors_bornes = fen.where((F.col("consommation") < 15000) | (F.col("consommation") > 110000) |
                        (F.col("taux_co2") < 0) | (F.col("taux_co2") > 200)).count()
controle("valeurs_hors_bornes", "fenetre", hors_bornes, 0, hors_bornes == 0)

schema = ("run_id string, run_ts timestamp, controle string, perimetre string, "
          "valeur double, seuil double, statut string, detail string")
spark.createDataFrame(resultats, schema).write.mode("append").saveAsTable("dq_resultats")

ko = [r for r in resultats if r[6] == "KO"]
print(f"{len(resultats)} contrôles, {len(ko)} en échec")
for r in ko[:30]:
    print("  KO :", r[2], r[3], r[7])